# Chapter 1: Common Conventions and API Elements of scikit-learn

**Reference:** John Sukup, *scikit-learn Cookbook*, Third Edition (2025).

This notebook follows the main topics of Chapter 1. The demonstrations are independently written and pair each code example with a short explanation of its output.

**Source:** John Sukup, *scikit-learn Cookbook*, Third Edition (Packt, 2025). The sections below follow the topics and selected examples presented in the book. Explanations and interpretations are summarized in original English.

## 1. Introduction to scikit-learn’s design philosophy

The library offers a consistent interface regardless of the underlying algorithm. A model generally learns with `fit()` and produces predictions with `predict()`. This example keeps the data fixed while switching between two regressors.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
X = np.arange(1, 9).reshape(-1, 1)
y = np.array([2.2, 4.1, 5.8, 8.0, 9.9, 12.1, 14.0, 16.2])
for model in [LinearRegression(), Ridge(alpha=1.0)]:
    model.fit(X, y)
    print(type(model).__name__, 'prediction:', np.round(model.predict([[9], [10]]), 2))

LinearRegression prediction: [18.05 20.06]
Ridge prediction: [17.84 19.8 ]


**Interpretation**

Both estimators are trained through the same method calls. Their numerical predictions are close but not identical because Ridge includes a penalty on the coefficient.

## 2. Understanding estimators

An estimator learns parameters from supplied data. In supervised regression, `fit(X, y)` estimates relationships between inputs and targets; `predict()` applies those relationships to unseen observations.

In [2]:
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X, y)
print('Coefficient:', model.coef_.round(3))
print('Intercept:', round(float(model.intercept_), 3))
print('Predictions:', model.predict([[9], [10]]).round(3))

Coefficient: [2.004]
Intercept: 0.021
Predictions: [18.054 20.057]


**Interpretation**

The coefficient represents the approximate increase in the target for one additional input unit, while the intercept gives the fitted value at zero. Predictions extend the learned trend.

## 3. Transformers and the transform() method

A transformer calculates statistics or a mapping during `fit()` and applies them during `transform()`. Scaling can prevent features with larger units from dominating downstream calculations.

In [3]:
from sklearn.preprocessing import StandardScaler
train = np.array([[2, 100], [4, 150], [6, 200]], dtype=float)
test = np.array([[8, 250]], dtype=float)
scaler = StandardScaler().fit(train)
print('Training values after scaling:\n', scaler.transform(train).round(3))
print('Test value using training statistics:', scaler.transform(test).round(3))

Training values after scaling:
 [[-1.225 -1.225]
 [ 0.     0.   ]
 [ 1.225  1.225]]
Test value using training statistics: [[2.449 2.449]]


**Interpretation**

The transformed training columns have zero mean and unit variance. The test observation is scaled with the training mean and standard deviation, avoiding test-data leakage.

## 4. Handling custom estimators and transformers

Custom preprocessing can follow scikit-learn conventions using `BaseEstimator` and `TransformerMixin`. This makes an otherwise ordinary Python class compatible with pipelines.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
class AddConstant(BaseEstimator, TransformerMixin):
    def __init__(self, constant=1.0):
        self.constant = constant
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return np.asarray(X) + self.constant
custom = AddConstant(constant=3)
print(custom.fit_transform([[1, 2], [3, 4]]))

[[4 5]
 [6 7]]


**Interpretation**

The operation increases each value by the chosen constant. Because the class exposes `fit()` and `transform()`, it can be reused as a preprocessing step in a scikit-learn workflow.

## 5. Pipelines and workflow automation

A pipeline combines preprocessing and prediction in one object. When its `fit()` method is called, each transformation is fitted in sequence before the final estimator.

In [5]:
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(StandardScaler(), Ridge(alpha=2.0))
pipe.fit(X, y)
print('Pipeline steps:', list(pipe.named_steps))
print('Predictions:', pipe.predict([[9], [10]]).round(3))

Pipeline steps: ['standardscaler', 'ridge']
Predictions: [16.25  17.853]


**Interpretation**

Scaling and regression are executed in a consistent order. Predictions use the scaler fitted on the original training inputs, rather than recalculating preprocessing statistics.

## 6. Common attributes and methods

Fitted estimators expose learned attributes, usually ending with an underscore. The `score()` method provides a convenient default metric, although its meaning depends on the estimator type.

In [6]:
print('Ridge coefficient:', pipe.named_steps['ridge'].coef_.round(3))
print('Ridge intercept:', round(float(pipe.named_steps['ridge'].intercept_), 3))
print('Training R-squared:', round(pipe.score(X, y), 4))

Ridge coefficient: [3.673]
Ridge intercept: 9.038
Training R-squared: 0.9592


**Interpretation**

The reported R-squared measures fit on these training observations, not generalization performance. Learned coefficients should be interpreted relative to the scaled feature representation.

## 7. Hyperparameter tuning with search methods

Model parameters such as Ridge `alpha` are chosen before fitting. `GridSearchCV` evaluates several choices using cross-validation, reducing reliance on a single train–test split.

In [7]:
from sklearn.model_selection import GridSearchCV
search = GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                      {'ridge__alpha': [0.01, 0.1, 1, 10]}, cv=4)
search.fit(X, y)
print('Best alpha:', search.best_params_)
print('Mean CV R-squared:', round(search.best_score_, 4))

Best alpha: {'ridge__alpha': 0.1}
Mean CV R-squared: 0.9378


**Interpretation**

The selected alpha has the strongest average cross-validation score among the tested values. With a very small dataset, those scores can vary substantially with the folds.

## 8. Working with metadata and API best practices

Scikit-learn stores configuration separately from learned parameters. Inspecting `get_params()` is useful for reproducibility, while controlled random states and proper data splitting help make experiments repeatable.

In [8]:
print('Configured alpha:', pipe.get_params()['ridge__alpha'])
print('Number of observations:', X.shape[0])
print('Model class:', type(pipe[-1]).__name__)

Configured alpha: 2.0
Number of observations: 8
Model class: Ridge


**Interpretation**

The output describes the pipeline configuration independently of predictions. Clear parameter management makes it easier to review and reproduce a modeling workflow.

## Book recipe: Understanding estimators — book regression example

The book introduces `LinearRegression` with a five-observation dataset to show how `fit()` estimates the relationship and `predict()` applies it to new feature values. This short example follows the book’s input values.

In [9]:
from sklearn.linear_model import LinearRegression
import numpy as np

X_book = np.array([[1], [2], [3], [4], [5]])
y_book = np.array([1, 2, 3, 3.5, 5])
book_model = LinearRegression()
book_model.fit(X_book, y_book)
X_new_book = np.array([[6], [7]])
print('Predictions:', book_model.predict(X_new_book))

Predictions: [5.75 6.7 ]


**Interpretation**

The values predicted for 6 and 7 extend the line learned from the five samples. Because the values are not perfectly linear, the fitted slope represents an average trend rather than an exact match to every observation.

## Book recipe: Transformers — book StandardScaler example

The book then illustrates `fit()` and `transform()` using a two-column matrix. Each column is standardized independently.

In [10]:
from sklearn.preprocessing import StandardScaler
X_book_scale = np.array([[1, 2], [3, 4], [5, 6]])
book_scaler = StandardScaler()
book_scaler.fit(X_book_scale)
print(book_scaler.transform(X_book_scale))

[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


**Interpretation**

The first observation has a negative standardized score, the middle observation is centered at zero, and the last observation has a positive score. This follows directly from the evenly spaced input values.